# Negotiate, Then Solve: reading policy documents with Unlimited-OCR

[Baidu Unlimited-OCR](https://huggingface.co/baidu/Unlimited-OCR) (3B, 0.5B active, MIT) parses scans and
photos into Markdown. It needs about 8 GB of VRAM, so it runs here rather than on the laptop (GTX 1050, 3 GB).

The notebook reads every PDF page and image in the attached dataset and writes one
`<file>.ocr.json` per document. Copy those files next to the originals in `data/policies/`:
`agents.documents` then uses them for every page without a text layer, and labels those rules
`ocr: baidu/Unlimited-OCR (precomputed)`. Pages with a text layer still use the text layer.

**Before running**

1. Zip `data/policies/` and upload it as a Kaggle dataset; add it with *Add Input*.
2. *Settings*: **GPU T4** (x1), **Internet on**.
3. Run all. A T4 has no bf16, so the model loads in float16 and falls back to float32 (about 12 GB)
   if float16 misbehaves. Check the printed sample before trusting a whole batch.

In [ ]:
%%capture
!pip install -q "transformers==4.57.1" "einops==0.8.2" "addict==2.4.0" "easydict==1.13" pypdfium2 pillow

In [ ]:
import glob, json, os, re, shutil, tempfile
from pathlib import Path

import pypdfium2 as pdfium
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

found = sorted(glob.glob("/kaggle/input/**/*", recursive=True))
DOCS = [Path(f) for f in found if Path(f).suffix.lower() in {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".webp", ".bmp"}]
OUT = Path("/kaggle/working/sidecars")
OUT.mkdir(parents=True, exist_ok=True)
DPI = 220  # the same resolution agents.documents renders scans at
print(torch.cuda.get_device_name(0), "|", len(DOCS), "documents:", [d.name for d in DOCS])

In [ ]:
MODEL = "baidu/Unlimited-OCR"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

def load(dtype):
    m = AutoModel.from_pretrained(MODEL, trust_remote_code=True, use_safetensors=True, torch_dtype=dtype)
    return m.eval().cuda()

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = load(DTYPE)
print("loaded in", DTYPE, f"{torch.cuda.memory_allocated() / 2**30:.1f} GB")

## Reading one page

`model.infer` writes its result into `output_path`; the text is taken from what it returns or,
if it returns nothing, from the Markdown file it saved. Grounding tokens
(`<|ref|>…<|/ref|><|det|>[[boxes]]<|/det|>`) are left in: `agents.documents` strips them.

In [ ]:
def read_page(image_path: str) -> str:
    out_dir = tempfile.mkdtemp(prefix="uocr_")
    res = model.infer(tokenizer, prompt="<image>document parsing.", image_file=image_path, output_path=out_dir,
                      base_size=1024, image_size=640, crop_mode=True, max_length=8192,
                      no_repeat_ngram_size=35, ngram_window=128, save_results=True)
    text = res if isinstance(res, str) and res.strip() else ""
    if not text:
        saved = sorted(glob.glob(f"{out_dir}/**/*.m*d", recursive=True))  # .md / .mmd
        text = Path(saved[0]).read_text(encoding="utf-8") if saved else ""
    shutil.rmtree(out_dir, ignore_errors=True)
    return text


def pages_of(doc: Path) -> list[str]:
    tmp = Path(tempfile.mkdtemp(prefix="pages_"))
    if doc.suffix.lower() != ".pdf":
        p = tmp / "page_1.png"
        Image.open(doc).convert("RGB").save(p)
        return [str(p)]
    pdf = pdfium.PdfDocument(str(doc))
    paths = []
    for i, page in enumerate(pdf, 1):
        p = tmp / f"page_{i}.png"
        page.render(scale=DPI / 72).to_pil().save(p)
        paths.append(str(p))
    pdf.close()
    return paths


sample = read_page(pages_of(DOCS[0])[0])
if len(sample.strip()) < 20 and DTYPE == torch.float16:
    print("float16 produced nothing useful; reloading in float32")
    del model; torch.cuda.empty_cache()
    DTYPE = torch.float32
    model = load(DTYPE)
    sample = read_page(pages_of(DOCS[0])[0])
print(sample[:1500])

In [ ]:
for doc in DOCS:
    pages = {str(i): read_page(p) for i, p in enumerate(pages_of(doc), 1)}
    (OUT / f"{doc.name}.ocr.json").write_text(json.dumps({"model": MODEL, "dtype": str(DTYPE), "pages": pages},
                                                        ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{doc.name}: {len(pages)} page(s), {sum(len(t) for t in pages.values())} characters")

shutil.make_archive("/kaggle/working/sidecars", "zip", OUT)
print("download /kaggle/working/sidecars.zip and unzip it into data/policies/")